# **Codenames με Μηχανική Μάθηση**  

Στο παιχνίδι **Codenames**, υπάρχει ένα πλέγμα από λέξεις, όπου κάθε λέξη ανήκει σε μία από δύο κατηγορίες: **καλές** (τις οποίες θέλουμε να εντοπίσει ο συμπαίκτης μας) και **κακές** (που πρέπει να αποφευχθούν). Εσύ γνωρίζεις ποιες λέξεις είναι καλές και ποιες κακές, αλλά ο συμπαίκτης σου βλέπει μόνο τις λέξεις χωρίς να γνωρίζει την κατηγορία τους.  

Ο σκοπός σου είναι να καθοδηγήσεις τον συμπαίκτη σου, παρέχοντας του **υποδείξεις** (hints), ώστε να εντοπίσει όλες τις καλές λέξεις με **τον ελάχιστο δυνατό αριθμό υποδείξεων**.  

## **Μηχανισμός των υποδείξεων**  
Κάθε υπόδειξη αποτελείται από:  
- **Μία λέξη** (word) που υπάρχει στο λεξιλόγιο.  
- **Έναν αριθμό $n$**, που δηλώνει ότι ο συμπαίκτης σου πρέπει να μαντέψει τις $n$ πιο σχετικές λέξεις *(που απομένουν)* με βάση την έννοια της υπόδειξης.  

Η σχετικότητα μεταξύ των λέξεων καθορίζεται από **προϋπολογισμένα word embeddings** από το Fasttext, τα οποία αναπαριστούν σημασιολογικές ομοιότητες μεταξύ λέξεων. Υποθέτουμε ότι ο συμπαίκτης σου είναι ένας ιδανικός παίκτης (optimal player), που κάθε φορά επιλέγει τις $n$ λέξεις που απομένουν και είναι πιο κοντά στην υπόδειξη με βάση τα embeddings.  

## **Στόχος**  
Ο στόχος είναι να βρούμε μια **στρατηγική** που να παράγει υποδείξεις με τέτοιο τρόπο ώστε:  
- Να **ελαχιστοποιούμε τον αριθμό των υποδείξεων** που απαιτούνται για να εντοπίσει ο συμπαίκτης μας όλες τις καλές λέξεις.  
- Να αποφεύγουμε υποδείξεις που θα μπορούσαν να οδηγήσουν τον συμπαίκτη μας σε κακές λέξεις.  

## **Παράδειγμα**  
Έστω ότι έχουμε το παρακάτω πλέγμα λέξεων:  


|  |  |  |  |  
| --- | --- | --- | --- |  
| **ΜΗΛΟ** | ΣΚΥΛΟΣ | ΑΥΤΟΚΙΝΗΤΟ | ΓΑΤΑ |  
| ΒΙΒΛΙΟ | **ΔΑΣΟΣ** | ΠΟΔΟΣΦΑΙΡΟ | ΤΡΑΠΕΖΙ |  
| ΧΕΙΜΩΝΑΣ | **ΚΑΡΟΤΟ** | ΠΛΑΖ | **ΒΟΥΝΟ** |  

Οι καλές λέξεις είναι: **ΜΗΛΟ, ΚΑΡΟΤΟ, ΔΑΣΟΣ, ΒΟΥΝΟ**.  

Μια καλή στρατηγική θα μπορούσε να είναι να δώσουμε την υπόδειξη **("ΕΛΑΤΟ", 3)**. Η πιο κοντινή λέξη στο ΕΛΑΤΟ είναι το **ΔΑΣΟΣ**, μετά το **ΒΟΥΝΟ**, μετά το **ΜΗΛΟ** και μετά ο **ΧΕΙΜΩΝΑΣ**. Επιλέγοντας, τον αριθμό 3 μαζί με τη λέξη ΕΛΑΤΟ καλύπτουμε τις περισσότερες καλές λέξεις αποφεύγοντας τη λέξη **ΧΕΙΜΩΝΑΣ** που δεν πρέπει να μαντευτεί.

Έπειτα, μπορούμε να δώσουμε την υπόδειξη **("ΛΑΧΑΝΙΚΟ", 1)** ή **("ΚΑΡΟΤΟ", 1)** για να κατευθύνουμε τον συμπαίκτη μας στο **ΚΑΡΟΤΟ**.  

Έτσι, αντί να δίνουμε μία υπόδειξη για κάθε λέξη, μειώσαμε τον συνολικό αριθμό υποδείξεων σε **μόλις δύο**, επιτυγχάνοντας τον στόχο του παιχνιδιού πιο αποδοτικά.  


In [3]:
import os
import numpy as np
import pandas as pd
from gdown import download

# 1. Κατεβάζουμε τα embeddings

In [4]:
def load_embeddings(id, filename="embeddings.csv.gz"):
    if not os.path.exists(filename):
        download(id=id, output=filename, quiet=False)

    df = pd.read_csv(filename, header=None, engine='pyarrow')
    words = df.iloc[:, 0].tolist()
    embeddings = df.iloc[:, 1:].to_numpy(dtype=float)
    embeddings /= np.linalg.norm(embeddings, axis=1, keepdims=True)
    return words, np.array(embeddings)

words, embeddings = load_embeddings('1biezzvCn3TkxRLy-7t6LA6M_bNFV8xl_')
word_to_index = {word: i for i, word in enumerate(words)}

print( "Πρώτες 3 λέξεις:", words[:3] )
print( "Το index της λέξης 'ρωγμή':", word_to_index['ρωγμή'])
print( "Οι διαστάσεις του πίνακα των embeddings:", embeddings.shape )

Downloading...
From: https://drive.google.com/uc?id=1biezzvCn3TkxRLy-7t6LA6M_bNFV8xl_
To: /content/embeddings.csv.gz
100%|██████████| 20.0M/20.0M [00:00<00:00, 121MB/s]


Πρώτες 3 λέξεις: ['κριός', 'ξένος', 'ρωγμή']
Το index της λέξης 'ρωγμή': 2
Οι διαστάσεις του πίνακα των embeddings: (32057, 300)


# 2. Ορίζουμε την ομοιότητα 2 λέξεων

Η ομοιότητα ορίζεται ως το εσωτερικό γινόμενο των embeddings.
Δηλαδή για embeddings `(α1,α2,α3,...)` και `(β1,β2,β3,...)` η ομοιότητα είναι:
```
α1*β1 + α2*β2 + α3*β3+....
```
Η μέγιστη τιμή της ομοιότητας είναι 1 και προκύπτει όταν συγκρίνουμε μια λέξη με τον εαυτό της

In [5]:
def similarity(i, j):
    return np.dot(embeddings[i], embeddings[j])

word1 = 'κόβω'
word2 = 'ψαλίδι'
word1index = word_to_index[word1]
word2index = word_to_index[word2]
print( "Ομοιότητα('{}', '{}') = {:0.3f}".format( word1, word2, similarity(word1index, word2index) ) )

Ομοιότητα('κόβω', 'ψαλίδι') = 0.408


# 2'. Ταυτόχρονος υπολογισμός

Για μεγαλύτερη ταχύτητα μπορούμε να υπολογίσουμε την ομοιότητα μιας λέξης με όλες τις άλλες ταυτόχρονα και αντίστοιχα και τις αποστάσεις

In [6]:
emb = embeddings[ word_to_index['ζωγραφίζω'] ]
sim = np.dot(embeddings, emb)

top10 = sim.argsort()[-10:][::-1] # Top 10 λέξεις
for i in top10:
    print( '{:0.3f}'.format(sim[i]), words[i] )

1.000 ζωγραφίζω
0.666 φωτογραφίζω
0.640 χρωματίζω
0.631 σκιτσάρω
0.616 σχεδιάζω
0.595 δημιουργώ
0.593 αποτυπώνω
0.591 μουτζουρώνω
0.586 σκαλίζω
0.583 ζωγραφική


# 3. Φορτώνουμε τα tasks και τη συνάρτηση αξιολόγησης

In [20]:
tasks = {
    "basic": [  # 3x4
        ('μήλο', 1),    ('σκύλος', 0),  ('αυτοκίνητο', 0),  ('γάτα', 0),
        ('βιβλίο', 0),  ('δάσος', 1),   ('ποδόσφαιρο', 0),  ('τραπέζι', 0),
        ('χειμώνας', 0),('καρότο', 1),  ('πλαζ', 0),        ('βουνό', 1)
    ],
    "easy1": [  # 5x5
        ("έστω", 0),      ("αμάχη", 0),     ("αφράτος", 1),   ("βάραθρο", 1),   ("βιδέλο", 0),
        ("γέρασμα", 0),   ("διαπάλη", 0),   ("εγγύς", 1),     ("ενδημία", 0),   ("ερύθημα", 1),
        ("ευθύνη", 0),    ("κίρρωση", 1),   ("κιλότο", 0),    ("κονάκι", 0),    ("κυρούλα", 1),
        ("μαμά", 0),      ("μπαμπάς", 1),   ("μόνιμος", 1),   ("νότος", 1),     ("οπάλι", 0),
        ("οπαλίνα", 1),   ("ορίζω", 1),     ("πετώ", 0),      ("τάδες", 1),     ("φραπές", 0)
    ],
    "easy2": [  # 5x5
        ("αγώνας", 0),    ("ανεβάζω", 1),   ("απωθώ", 0),    ("αφράτος", 0),   ("βίδα", 1),
        ("γίδα", 1),      ("γεράνι", 1),    ("δηνάριο", 0),  ("καίσαρ", 1),    ("καρσί", 1),
        ("κατιμάς", 1),   ("καφέ", 0),      ("κόλλυβο", 0),  ("λέρα", 0),      ("μονιμάς", 0),
        ("μορς", 1),      ("μοτέρ", 1),     ("μποφόρ", 0),   ("ομού", 0),      ("ορφικός", 0),
        ("παράγω", 1),    ("σήραγγα", 1),   ("σεισμός", 1),  ("υπατεία", 1),   ("ωστόσο", 0)
    ],
    "hard1": [  # 6x6
        ("Παναμάς", 1),   ("άζυμος", 1),    ("άλλοθι", 1),   ("άμπελος", 1),   ("άχρονος", 0),   ("έθιμο", 0),
        ("ακοή", 0),      ("γαρμπής", 0),   ("δέων", 1),     ("δούλεψη", 0),   ("επίπεδο", 0),   ("θάρρος", 0),
        ("κάποιος", 0),   ("καλπάζω", 1),   ("καφέ", 0),     ("λόχος", 1),     ("μαυρίζω", 0),   ("μπράιγ", 1),
        ("μόρβα", 1),     ("νυχτικό", 0),   ("ορμόνη", 1),   ("περάτης", 1),   ("πλάστης", 1),   ("πρόταση", 0),
        ("ριζά", 1),      ("σάρκωμα", 0),   ("σαλτάρω", 0),  ("συστάδα", 1),   ("σχόλιο", 1),    ("ταβέρνα", 1),
        ("ταχύς", 0),     ("τορπίλη", 0),   ("τούντρα", 0),  ("φυτό", 0),      ("χρωστώ", 1),    ("ώμος", 1)
    ],
    "hard2": [  # 6x6
        ("ακαθισία", 1),   ("αναδιανομή", 0),   ("γονυκλινής", 1), ("γουναρικό", 0),       ("δεξίωση", 1),        ("δωρεοδόχος", 1),
        ("είναι", 0),      ("επιεικής", 0),     ("θεματικός", 1),  ("κατοστάρικο", 0),     ("κοίλανση", 0),       ("κτίση", 1),
        ("λευκόχρυσος", 1),("ναυς", 0),         ("ομιλητική", 0),  ("οργανωτής", 1),       ("πάτρονας", 1),       ("πέραμα", 1),
        ("παράθεμα", 1),   ("παρουσιαστικό", 0),("πεντάγραμμο", 0),("πλατωνιστής", 0),     ("πλιατσικολόγος", 1), ("ραδιοαστρονομία", 1),
        ("σαλβάρι", 1),    ("σκάφος", 0),       ("σκωρίαση", 0),   ("στρουθοκαμηλισμός",0),("σφουγγαρίστρα", 0),  ("τάισμα", 0),
        ("υαλοστάσιο", 0), ("υδρωπικία", 0),    ("χαρούπι", 1),    ("χιλιετία", 1),        ("χρησικτησία", 1),    ("ψυχογλωσσολογία", 1)
    ]
}

def evaluate(task, hints):
    ids = [word_to_index[w] for w,_ in task]
    good = [i for i,(w,g) in enumerate(task) if g==1]
    covered = []
    emb = embeddings[ids]
    for (w,n) in hints:
        s = emb @ embeddings[ word_to_index[w] ]
        closest = s.argsort()[::-1]
        cnt = 0
        for i in closest:
            if i in covered: continue
            cnt += 1
            if i not in good:
                return ( f'ΣΦΑΛΜΑ: Η υπόδειξη ("{w}", {n}) συμπεριλαμβάνει την κακή λέξη "{words[ids[i]]}" σαν {cnt}η.' )
            covered.append(i)
            if cnt == n: break
        if cnt < n:
            return ( f'ΣΦΑΛΜΑ: Ο αριθμός {n} είναι μεγαλύτερος από τις λέξεις από όσες απομένουν.' )
    if len(covered) < len(good):
        return ( f'ΣΦΑΛΜΑ: Οι λέξεις {[ task[i][0] for i in good if i not in covered]} δεν καλύφθηκαν.' )
    return f'ΣΩΣΤΑ: Οι λέξεις καλύφθηκαν με {len(hints)} υποδείξεις.'

In [66]:
def codenames_solver(task, words, embeddings, word_to_index, max_hint_size=3, top_k_candidates=30):
    task_words = {w for w, _ in task} #all task words

    label_map = {w: label for w, label in task}
    good_words = {w for w, label in task if label == 1}  #target words
    bad_words = {w for w, label in task if label == 0}   #bad words

    remaining = set(good_words)  #words left to cover
    hints = []  #store generated hints

    while remaining:
        inds = [word_to_index[w] for w in remaining]
        embs = embeddings[inds]
        center = embs.mean(axis=0) #avarage embedding of the good words
        center /= np.linalg.norm(center)  # Normalize

        sims = embeddings @ center
        sorted_indices = np.argsort(-sims)  #find closest words

        candidate_indices = [idx for idx in sorted_indices if words[idx] not in task_words][:top_k_candidates] #keep top k candidates who arent in the task words

        for idx in candidate_indices:
            word = words[idx]  # Candidate hint word

            #similarity of hint with all words sorted
            hint_emb = embeddings[idx]
            sims_hint = embeddings @ hint_emb
            sorted_hint = np.argsort(-sims_hint)

            hit_count = 0  #number of good words covered
            bad_flag = False  # bad words flag
            covered = set()  # Good words covered by this hint

            for neighbor_idx in sorted_hint:
                neighbor_word = words[neighbor_idx]
                if neighbor_word in bad_words:
                    bad_flag = True
                    break
                if neighbor_word in remaining:
                    covered.add(neighbor_word) #good words it covered
                    hit_count += 1
                    if hit_count == max_hint_size:
                        break

            if not bad_flag and covered:
                hints.append((word, len(covered)))  # add hint and how many it covers
                remaining -= covered  # remove covered words from remaining
                break
        else:
            #if it didnt find any good hints for the remaining words, find one hint per remaining word
            word = remaining.pop()
            idx = word_to_index[word]
            similar = embeddings @ embeddings[idx]
            sorted_similar = np.argsort(-similar) #sort similarities

            for idx in sorted_similar:
                simword = words[idx]
                if simword not in task_words:
                    if simword == 'μπαμπέσης':
                        print("wtff")
                    hints.append((simword, 1)) #append the closest word that isnt in the task words
                    break

    return hints



In [73]:
t = "hard2"
tt = tasks[t]
hints = codenames_solver(tt, words, embeddings, word_to_index, max_hint_size = 2)
print(hints)
result = evaluate(tt, hints)
print(result)

task_words = [w for w, _ in tt]
print(task_words)

[('αγοραστός', 2), ('λαμπίκος', 2), ('ψυχοβιολογία', 2), ('κληροδόχος', 2), ('τελετάρχης', 2), ('επισκοπεία', 2), ('μαφόριο', 2), ('άργυρος', 1), ('νομιμόφρονας', 1), ('πλάση', 1), ('κείμενο', 1)]
ΣΩΣΤΑ: Οι λέξεις καλύφθηκαν με 11 υποδείξεις.
['ακαθισία', 'αναδιανομή', 'γονυκλινής', 'γουναρικό', 'δεξίωση', 'δωρεοδόχος', 'είναι', 'επιεικής', 'θεματικός', 'κατοστάρικο', 'κοίλανση', 'κτίση', 'λευκόχρυσος', 'ναυς', 'ομιλητική', 'οργανωτής', 'πάτρονας', 'πέραμα', 'παράθεμα', 'παρουσιαστικό', 'πεντάγραμμο', 'πλατωνιστής', 'πλιατσικολόγος', 'ραδιοαστρονομία', 'σαλβάρι', 'σκάφος', 'σκωρίαση', 'στρουθοκαμηλισμός', 'σφουγγαρίστρα', 'τάισμα', 'υαλοστάσιο', 'υδρωπικία', 'χαρούπι', 'χιλιετία', 'χρησικτησία', 'ψυχογλωσσολογία']


# 4. Υπολογιζουμε τις κατάλληλες υποδείξεις για κάθε task


In [68]:
#for max hint = 2
import json
all_answers = {
    "basic": [('δέντρο', 2), ('πορτοκάλι', 2)],
    "easy1": [('ολοστρόγγυλος', 2), ('παλιόγερος', 2), ('στένωμα', 2), ('παπαλίνα', 2), ('μέλλοντας', 2), ('σπλήνας', 2)],
    "easy2": [('θελιά', 2), ('στρόφαλος', 2), ('μετασχηματίζω', 2), ('αστραποβόλημα', 2), ('μονοκράτορας', 2), ('σεισμολογία', 2), ('σήμα', 1)],
    "hard1": [('χλωρός', 2), ('χαράκι', 2), ('συστοιχία', 1), ('άρτος', 1), ('στρίγκλισμα', 2), ('ήσσων', 1), ('ουλαμός', 1), ('μαλάρια', 1), ('ευγνωμονώ', 1), ('γοφός', 1), ('γραφή', 1), ('ψαροταβέρνα', 1), ('παναμάς', 1), ('κείμενο', 1), ('τεστοστερόνη', 1)],
    "hard2": [('αγοραστός', 2), ('λαμπίκος', 2), ('ψυχοβιολογία', 2), ('κληροδόχος', 2), ('τελετάρχης', 2), ('επισκοπεία', 2), ('μαφόριο', 2), ('άργυρος', 1), ('νομιμόφρονας', 1), ('πλάση', 1), ('κείμενο', 1)]
}
json.dumps(all_answers)

'{"basic": [["\\u03b4\\u03ad\\u03bd\\u03c4\\u03c1\\u03bf", 2], ["\\u03c0\\u03bf\\u03c1\\u03c4\\u03bf\\u03ba\\u03ac\\u03bb\\u03b9", 2]], "easy1": [["\\u03bf\\u03bb\\u03bf\\u03c3\\u03c4\\u03c1\\u03cc\\u03b3\\u03b3\\u03c5\\u03bb\\u03bf\\u03c2", 2], ["\\u03c0\\u03b1\\u03bb\\u03b9\\u03cc\\u03b3\\u03b5\\u03c1\\u03bf\\u03c2", 2], ["\\u03c3\\u03c4\\u03ad\\u03bd\\u03c9\\u03bc\\u03b1", 2], ["\\u03c0\\u03b1\\u03c0\\u03b1\\u03bb\\u03af\\u03bd\\u03b1", 2], ["\\u03bc\\u03ad\\u03bb\\u03bb\\u03bf\\u03bd\\u03c4\\u03b1\\u03c2", 2], ["\\u03c3\\u03c0\\u03bb\\u03ae\\u03bd\\u03b1\\u03c2", 2]], "easy2": [["\\u03b8\\u03b5\\u03bb\\u03b9\\u03ac", 2], ["\\u03c3\\u03c4\\u03c1\\u03cc\\u03c6\\u03b1\\u03bb\\u03bf\\u03c2", 2], ["\\u03bc\\u03b5\\u03c4\\u03b1\\u03c3\\u03c7\\u03b7\\u03bc\\u03b1\\u03c4\\u03af\\u03b6\\u03c9", 2], ["\\u03b1\\u03c3\\u03c4\\u03c1\\u03b1\\u03c0\\u03bf\\u03b2\\u03cc\\u03bb\\u03b7\\u03bc\\u03b1", 2], ["\\u03bc\\u03bf\\u03bd\\u03bf\\u03ba\\u03c1\\u03ac\\u03c4\\u03bf\\u03c1\\u03b1\\u03c2", 2], ["